# IOAI — 2026 Contest Audio Event Detection (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, urllib.request, gzip, shutil
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-audio-event-detection'
for f in ['X_test.npy','train.csv','test_ids.csv']:
    if not os.path.exists(f'data/{f}'): urllib.request.urlretrieve(f'{BASE}/{f}', f'data/{f}')
if not os.path.exists('data/X_train.npy'):
    with open('xt.gz','wb') as out:
        for part in ['Xtr_part_00','Xtr_part_01']:
            urllib.request.urlretrieve(f'{BASE}/{part}', part)
            with open(part,'rb') as p: shutil.copyfileobj(p, out)
    with gzip.open('xt.gz','rb') as g, open('data/X_train.npy','wb') as o: shutil.copyfileobj(g, o)
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Audio Event Detection — 모범답안 (처음부터 2D-CNN)

log-mel 을 1채널 이미지로 보고 **작은 CNN 을 처음부터** 학습한다(사전학습 없음). 주파수-시간 패턴을
conv 로 잡아 8클래스 분류. 가중 F1 ≈ 0.6~0.7 (최빈 대비 큰 향상, 참고점수 0.70).

In [ ]:
import numpy as np, pandas as pd, json
Xtr = np.load("data/X_train.npy").astype("float32")   # [N,64,300]
train = pd.read_csv("data/train.csv")                 # id, label
Xte = np.load("data/X_test.npy").astype("float32")
test_ids = pd.read_csv("data/test_ids.csv")["id"].tolist()
CLASSES = ["cough","sneeze","laughter","cry","dog_bark","siren","noise","none"]
print(Xtr.shape, "| classes", train.label.nunique())

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(0); dev="cuda" if torch.cuda.is_available() else "cpu"
CID={c:i for i,c in enumerate(CLASSES)}
mu = Xtr.mean((0,2),keepdims=True); sd = Xtr.std((0,2),keepdims=True)+1e-6   # per-frequency 정규화
def norm(X): return ((X-mu)/sd)[:,None,:,:]        # [N,1,64,300]
Xt = torch.tensor(norm(Xtr)); yt = torch.tensor([CID[c] for c in train.label])
loader = DataLoader(TensorDataset(Xt, yt), batch_size=64, shuffle=True)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.h = nn.Sequential(nn.Dropout(0.3), nn.Linear(128,8))
    def forward(self,x): return self.h(self.f(x))

net=CNN().to(dev); opt=torch.optim.Adam(net.parameters(),1e-3,weight_decay=1e-4); lossf=nn.CrossEntropyLoss(label_smoothing=0.05)
for ep in range(35):
    net.train()
    for xb,yb in loader:
        opt.zero_grad(); loss=lossf(net(xb.to(dev)),yb.to(dev)); loss.backward(); opt.step()
print("trained, loss", round(float(loss),4))

net.eval()
with torch.no_grad():
    logits = net(torch.tensor(norm(Xte)).to(dev)).cpu()
pred = [CLASSES[i] for i in logits.argmax(1).tolist()]
json.dump({str(test_ids[i]): pred[i] for i in range(len(test_ids))}, open("submission.json","w"), indent=2)
print("saved submission.json", len(pred))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)